# FinDisputeEval CFPB Seed Source EDA v05.1

This corrected run preserves the v05 dual-frame contract while adding typed byte-literal failures, bounded redaction measures, timezone-safe monthly grouping, and direct persistent output to Google Drive.

Population reference statistics use only `population_eligible == True`. Enrichment remains candidate-supply and stress-test material and never contributes to population prevalence.

## Runtime and persistence

In Colab, this notebook creates `/content/FinDisputeEval`, copies the three frozen CSVs from `MyDrive/FinDisputeEval`, attempts to preserve the prior v05 run if it still exists, reconstructs checksum-verified pipeline modules, and writes all v05.1 outputs directly to Google Drive.

In [ ]:
from pathlib import Path
import shutil
import sys

INSTALL_DEPENDENCIES = True
RUN_LANGUAGE_ID = True
RUN_PRESIDIO_SAMPLE = True
RUN_MINHASH = True
RUN_DEPENDENCY_PARSER = True
RUN_ID = None

IN_COLAB = "google.colab" in sys.modules
RAW_SUBDIR = Path("dataset/external/cfpb/raw/seed_build_v03_cache_2026-07-04")
RAW_FILENAMES = [
    "cfpb_zelle_fulltext.csv",
    "cfpb_inscope_recent.csv",
    "cfpb_prepaid_alltime.csv",
]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/FinDisputeEval")
    DRIVE_SOURCE_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
    PERSISTENT_OUTPUT_ROOT = (
        DRIVE_SOURCE_ROOT / "outputs/cfpb_seed_source_eda/eda_v051"
    )
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    raw_destination = PROJECT_ROOT / RAW_SUBDIR
    raw_destination.mkdir(parents=True, exist_ok=True)
    for filename in RAW_FILENAMES:
        destination = raw_destination / filename
        if destination.exists():
            continue
        matches = [
            path for path in DRIVE_SOURCE_ROOT.rglob(filename) if path.is_file()
        ] if DRIVE_SOURCE_ROOT.exists() else []
        if matches:
            shutil.copy2(matches[0], destination)
            print(f"Copied raw source: {matches[0]} -> {destination}")
    missing = [
        name for name in RAW_FILENAMES if not (raw_destination / name).exists()
    ]
    if missing:
        raise FileNotFoundError(
            f"Missing frozen files under {DRIVE_SOURCE_ROOT}: {missing}"
        )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(
        (
            path.resolve() for path in candidates
            if (path / "src/findisputeeval/cfpb_seed_source_eda_v051.py").exists()
        ),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Run from the FinDisputeEval repository.")
    PERSISTENT_OUTPUT_ROOT = None

print(f"Project root: {PROJECT_ROOT}")
print(f"Persistent output root: {PERSISTENT_OUTPUT_ROOT}")

In [ ]:
import hashlib

PREVIOUS_V05_RUN_ID = "20260713T092549Z"
if IN_COLAB:
    previous_source = (
        Path("/content/FinDisputeEval/outputs/data_pipeline/")
        / "cfpb_seed_source_eda/eda_v05"
        / f"run_{PREVIOUS_V05_RUN_ID}"
    )
    previous_destination = (
        DRIVE_SOURCE_ROOT
        / "outputs/cfpb_seed_source_eda/eda_v05"
        / previous_source.name
    )
    if previous_source.exists():
        shutil.copytree(
            previous_source, previous_destination, dirs_exist_ok=True
        )
        for source_file in previous_source.iterdir():
            if not source_file.is_file():
                continue
            destination_file = previous_destination / source_file.name
            source_hash = hashlib.sha256(source_file.read_bytes()).hexdigest()
            destination_hash = hashlib.sha256(
                destination_file.read_bytes()
            ).hexdigest()
            assert source_hash == destination_hash, source_file.name
        print(f"Preserved and verified prior v05 run: {previous_destination}")
    elif previous_destination.exists():
        print(f"Prior v05 run is already preserved: {previous_destination}")
    else:
        print(
            "Prior v05 runtime directory is no longer available. "
            "The executed notebook retains rendered results, but the 23 files "
            "must be regenerated from frozen inputs if they were not copied earlier."
        )

In [ ]:
import base64
import gzip
import hashlib

MODULES = {
    "cfpb_seed_source_eda_v05.py": (
        "6654878185eeca9b510537d0b7756c968b7329f7eb0947e86b91250e9719090c",
        "H4sIAAAAAAAC/9V9+3bbyNHn/3wKBJv5BhxTHF0sX3SiOB6PJ+OTuXhtJ/ttKAYHJEEJMQkyAGlZFvWdfY19vX2S/VX1vdEgKY9zzm5OxiKA7urq6u7q6rp1HMcvfnj9XVTn+eSgXqyrcR69/P559OHwNLouVlf4sMyqbJVHy8VyPctWxaKMsnIS5WVVjK/mebmKplU2z+t+p/Puqqij+WKynuXReFGusqKso9VVHk3yVV7Ni7KoV8WYq1f5us5GKAfoqzpaTLnci8UsG0XlYpWPFov3/c6rVVTmH/IKAGYAUUcSw2pxjd/FDK3PbvrR8/VqMQeOk2hWlJdr08qyKj5k45toWuSzSR1lVd4Z430xoQ4BuVW1ph7U0bpcFTNCoaii/ONyURGwbD0p0Ds0U0dX2Yc8GuV5Ccw/FPl1Pul34jjudKbVYh6l6XS9Wld5mkbFnGqjdfSCqVV3OupdvVI/r7L6alaM1OM/60WpfqMjV+o38JW/1mUxXkxyIJ6JJseL2SwfcwOqzRcLdCOvxHcqOZ5ldZ3r71k9KcarnvnUE4TRFfJVMc9VafXci+jfT4syF+WWwA+oq2KvCV3+sLpZgvrq/fPyphe9Ajo0ypoE5Xq+vAEiUblUr5YYELzA/5eTTqfz5uV//+urNy+/T1/8+tNff/7lbXQeDToR/hd/T4NW5eO8+JBP4p54+brCdBuv1OPb9ehg6b56Vdfr3P5e2C9egH6YAxXoOV/OMGGBY1ZhwqMRU2YOJG+i5Xo0w8Sq8nqJWv5n3QSGXX/7+6vXEY2ben6XXdYWLvNiRRPtQ5Gpl9zJmlbVasE4WZAVHgoBUUTgrxvAWM1MkWd2XdG9V9/j3bDTefvrX9+8eJm+ff3yBRH5VpT7lGNaYTbPZqv84yo+k+/5G62EEksdb+PxdDlK3cL9cf1BNsfFq8WMixpO4XzN/7XOa/Q+pYmWgjlQ2ePDo8cHh48ODo+2lc0+irLHKIjix3ZZrF4sCxQlHoFiR0/Sw6OTUIH6Kjs+fUSQHo8ePTk9PHoyejh++PDoNDudPj4+epqNstPH48noUf740fjp8fj05Onx6cNpNs1HJ+PDR0+PUDjLJo+mT2T7d5LWYCzjxTJPaa6Wu2joFm6joWG++9Dw+BQE/JI0fPo4ffjk6XYijkdH49HjSX5yfPToaX7y9PHx6eTk9NHjxydHTydPppPs6eT06fT0OM8Pn0weHU4ePxmNnz46fpifTI8mIK5HxGWFjaeYpBkmV8H02kZFr/QXmopHB0fHX5KMR+nxox1UfPhonD2cjkCXw6dPT/OT0cPDk+np5OjkZDI6fXKML0cg7KOHJ5MTzMLp06fTxydPnkzHk5PpdHT6cGSoeAdm+ifN6jv8L23t4HjT4vKMi4FV/hMYAMHF6kywcnpdZdfppKjEm2gT/QLmDxZBf/j7Yr1aru1agTLVukyLyVmEPTb0FUx/MU9J6jiLiCudR0TDw8dHJ/x9lmEfzy5BYDDPWUp7ztaWZtQUZIYZPryr1uYDJkZdTIpFWmdggHmzkFcgrYtPuUYpPTw81KAwL2jbboKQH9LVFWBdLWZAZTpbZAThsP/k1CmDDTBdQhZSTRwdP3G+42+6WrzPy9rQxSmAaV6l42xJUNLJYswCTEtZzE7wFeohyvCWJqFqIp4eOl2c5Mu8nOTl+AbtVHVeNTtbLyFQiWERg3tOyyodLyD9XOejtJ7HXG41LSZTxkAhqXv00DTKIhavD+4P4GWrtSbOqSg0zebF7IZJw4O1ztPxFdAz8PxiLgWfdPj7JJ9CUFsu6hWYLlpNkzqfTbvRwR+ZGmd6XdLrvr00AIPmXtL40O0DocXsQ550de1iKgDIRRRBJHbB6yZUifMocT4GcWiU+DaKaXXX+SoOfcOGnFdlNgt+JJ4Z/ACUgu9poaajdTGbpB8OTzD/xld5qrneQ7eOoUU+q7d33JBVvttGUYvtbKGqXeqzKSuA1HEb1dNlscxx2Mhbycu8LRXnlRRye3iQJhnoeXpP8rk9NCS03m+dmMyZt81LUeBcHwH65eI6UaeA/no17vaxUqf0Jom/+p9fzb+avPvqx69+/urt3+NmewFevqXxUOnzwJIE9Vb5fBnTDy5a809sBP2jx4/609WneBcdw00Zega+23RlgH8CUuBcqxvNYuQoYC4b/kIwDQJVjrNi2TJw6MQ0phG4tcbiLg5t5q/lHOQThwA/Fpu72ecFz+ZZCH5I578Bhq6Hs1YfR43sBzq6D9FrPggmwD9bz1bpNBuvFtXNOZXvWiDAOD+Ak+PTmQMhuMNfpzi0LsqdJVEKR3yM0K6CEgfqI85htJ/sqKD2QX3m310Fh6l5VhW/jVi8q/0mCNM8Y31ClV9Cm0HkNrDML5yxh3sN3opm8WxxeeZV3lG306EJjfWeEgPgmYyqYqbJSbyFRRT1YrqooM5IFCQh5qYkvSdGpOtF46t1+d6RvI6iP/wB0ozb5KS4hOiNr1J70hfwJItjbRVB7WNBlklcjeIuaRVQPc/mZvVdX6H5aDRbjN9HZ+fyM5Z1NkkMHl2XXYiW++sldTfhul2HCuL7Vf5R/PJ7TMfjhP5hkSlISK9P+MwVun1IY2A/SbxeTQ+exN1uoJX0nwsIPVhHOJskH7LZmqav0r3QaA+DTcabuE81k5r1XcktNcrVuxFGLuKfGBDxo2aGrkvcdVXreObFpTa7GrLBPEtYK+iut55awiV/UqQwOwE0an+T0FgfKNV9pEusMDPFGL/GakbXhBby4xKsoCCtBmZRPn5f90krx9uPUS+hdN9SM2UdyR3qmhRWmAOi/xClEl/9BAQjes+96UPptp6XdVcMPughYVi8PSugmPkbkehlVS2qZBrfWp2+O9PN0kmyqKD9kUBxwJWf7uQmKggJ/JaZoeFbQV3d4K2ofkaFXvDPhJd3Ca0MDf85Se493Ro/iuEVFWl8/U7fmTMqYIEVnP+QYQs1b8eLHD0SoPVL6t/5VAg/Tp/lmdTuVF9NGkHYHg58n24Eboq0/GHgKq6G/fxfSRx3+1CDJd37UH2Ezfx9ZAOTszre0eBkTTOMlMtJo1n9DXvSmvkWqiU74bzP86WgKOSo9Tzpdu/RkVuv0TuhDGf9tP7kdJS6KFYqDqVKJK2ThqjAa3EFEPmgZePyHvWa/Qlweb3SCTgHIRefoCiXi1foz4U2nhSzeZSpT1qa0It2D0nl9k4MmKrLp8czaP7r1aC5NQ6GYj/FfLfoCEYEzQtNfVsL2i8gUdb2ANsioaCXPqdATiMYA6OLGtqSL7T/YjPKPwIxB6gZ5x9Q9ZfF6geM40QN98+SPThExLATsDtLutabob+pOgK4LPW7c4msr3EahtCypl/j0DKN3/74/ABViY9hbx9fMW0Zvb6coqqR6JYb/dpr9OvhXS+6BH1uBXZ3bccfXkjE/ia8PadQ6rkIUaM9d6OGESI/j4lrlZex+403Urw+F1vpQV34JWhhpkoUKjOf67FianGdzvM5yUjeV3CHmlpPVOumI9t2SGdPdMYO00oU6QbGj1WKnzF6bxbXB4JZfcb4UaNm9Ax+O0dQ7Z7Lm8T7OIhl9yG6VDfxkHdjTQ+/LBNRmYCEimrCdTBFVotUCaNux2Vd13wEXpYTjerzWGxmcY8EXblbCrn1PJ4XH1G20+yY2S7CKEH4LbPmhrHvKPk832wrX4fa+1q3J7aTO5yp+AtbWPng0zZCLhvtZ0tSALoo3TYQdMfszOGszcJS+S4mMD8MA6VoAsYsEwouFiiiteSCb4RKQHRPRzfU3zPBfnH0WRFdVizWB7Fj3byZzIEyQrJOtYkQJ3KqE97oSymHd1saS0mITY2RkY88zUlgzdytNsohqWLov2KJfpY5i0c8DRoggxiFbB9yqJqfhntAYItIGAI+hSBAsl9nMxuB7WsLZdBVlh6dU+Yu0IzZDtDZxz1A3wU4gpRcBtZaGCrOZx+6ZDlXrkncZajkNaFwnVAPhI9FspeQ1OtYopx7/mp/MtLcd9RqRHoRy+VDcHKS4spFebCAwmYGXkGCStMRRAlzmK4jqKcmgj9DfoJomkjk+0L6BmWj4rIk40GBc9pH6wiAXXYEtdBVsawdPa6C2r+sFuvl6CbxbNt8mpPitb+7aCD97PIysY/NZiAha9DgJaqqwUPu6OJfSQIhhP87MNR8pBUzblojpfV/pBtL5aFSe0+0elAEvChaPCka3hRBj4p9vSoCrhNDm7CEuiYkFC4D9bDnyWpoixvYrhXYPpTGq5sza7CkNlHMUrMkJQXPB/7oxaoKZr+iczwUDbmaZr1vyCmicQhPjO7AHz6zm0ygZlYSabeBPHMNc9pRJx57w+rROZFOPA5SdOSpqHLS0HhNp3kl1CMDqSlwlQaNqQYi004GaLICtE5/jI6GDlw+k0jQZw1+7fQnKIxoocQl3ZnX1+AYnRmNl8ahe9fcJveZGxaecunB12xRFasbOqI2XUAOewHfmqNeyM/h+M7hnoNYHRtUCyzxms8uA8EWtkxUSX0mAV26jrI9zLCIMaWSMzfnfRMPizfRBE31UqwbPI+XJg7LVb2KvWp6qTVbsEpay7nRFfVTDK61c/Swj517q2x7ZZev76yvye+N+NACPAhsJUJsU06SZiiqOHl29o/NxabrAsTbi83m992YVHmXcqMMEEO35U22346PC/De+Pjz/Lcj5EHcHyMj1qSwn10Wo1kLOv6YCisESY43eVY5NVrkycmqT0U9BMhwuagg1Y3W4/e5GJxy2b++yitweg3/j+wUc9oTLl3pcrauY8ExyPR+GneNXwNLXix0Ev9lrfvbnMxYruJfcmDy6WA+3dZR/8gq7QUW4QgBRmgrXH8SdlmCbBRrzI2W9gU4VdoInqlctPdEJthIc7uRzcK3GQI6uNvMarlp1UfL3GhgjM+tsWvUI8lBNQV7MNxc0FNrorY02e00BkmSpwVff3WS1xNtkjyDnGVA++/sJjFTqxdlUGaeH7VougxQa2SguINvs5rhMGlDk2RIPHD60o4Vq/23DYN3QNsL0nYq3w/idqJ/DnZbZrkFzjpOu7SMLUwEwHpNw+lpPD1SzMFrUyn1S5x6rWAVhtsAY7kHehIHOyAVr+exYy8KEOs6Ly6vPK65g7v3oqM+RDAUL7NSnwlI6aMqspL1+PA0PX3ydKth6aXSjqJwD4WlQO2aXCwVqW7grHdnjE0kKe/AWSrzCC/pgLsFr/i1f1afr6H4l/snVLrQhMAvHIB6ANSwD1kMRAmDPcMIyIFwJQ7ofBARp/SekY+hr3j+869//eVd+uYlRqUiXe98SRaJKr64vai/ufg9/hlcTHr94QN6JEr8/PztX2Bz/P75u5eBav95e3z3Lf+DjR1/e0/uNhcT/H14t6Hd8Y9io++/6iJkACasd69+/SXU+ojq3l3AGaDz4tc3sPS8bCuaPCvq7sUocGQFxmf4L+k/6ALM65+ev3j5468/ff/yTajBAfAVGL38fkOd2wjSdC+GqPw/fn3zfRDNwfODv2cHnw4Png4fAMLg6//zv/73wdB+233Gvfjh1zc/pz++fB5qvqMkpGLe/QegyACCTTZmcXajogU2MD8Us002mWBo680EXpVkiWbfpjXNoU0sQQWosXmfI6IG86ne0G5NHkibSVHDg4liZubc0PVVRkEudIDLJ12iX9zpdn56+e4dsP75+Zu/7IF8vR6Ru9emAngM+AbRDtdXiDMpKEbmhibfGB6OG1jL8BdBDxsKekAF2uFvqM3BWdQbinb//PynFE3/3NboxQgN/vWi/+wt/nuB/zYvfniz+eHFm+ebH75/8Rr/vvju+eblD++eb97kl2qhDV7+fbghDTO6rikmnqHK23woFqLgZpZfZrMo43CdTbbC1zK/2cyy63pdrDbLdVWvMw75wPyT3I+mNhR5ygenvK5wKCHVNnydybNj5vmTCG0fKwPJ4UOwCuHBwU4uwpuko2z6GBnSy3EBpT3uWLZLNM/WIczNf4ySwdcXkJf631wc/T7u6fqwk8wQz3KOsm+7Dfc2Bg1xFdBSkiSwZxBbE87B1Y1tZCVhGchAqujLzqU5qieqIcfuUtQUuJVh0BNRsxexxj8szoki/UnuONH0SKdPFAXwFO+eCLTyj+N8uYqStzfgmh+ZsfYsJtuL/iqisL5naPyuvd9yyKZYaMqGtL2EGGmMErm6CIdqvebaB5vEXDnawpx3Hnl8DlJXVmEk9SQw3itUodEBfitUVMlRV02OnuVXLzXZ2kVJF2H1lOwInO4WJQkKZJPZ5gClJqkV4gZvMujdqWYS//LDCzHnbORVHTF/IWDMMsyH+KK6KOmIhH+79lv1zqmrdyxstKMkHojnIcoKDyensLtZyRr01FLe3pNkabUptNTAKqJyVTyILlbDB4RytKPkRXl7AqGCO8cdtwrbc00vcDEyPMAkSFEg42ePyl/cYYGO9jqvEhdTd6sUdIgGPCwUJ5FXw51dnAhKDBA4Mdqj/OAf2C4vBhfDKEhCSRUDvvZKeaSawX2DZN1tlNoCM9h/h2x+g9iSebFwi3XIiY4bt18oLGoIzyFbu+Z0TCcqtdOkSDrEwJbTtVoaxLpCqluQ50f5xGBm2Xw0ySJyrTnjfweHw5YDpAJsN5nydlrvBfioDTDxUqvzQcQZZpj5tnWbSkldLG+Xn9VZbkq2S/I/b5WSklvgWn2lHaANRYrnGO4/7lsxtTquWLsD2ycMY97YBbr7NsJcqq0BgwC34nC0vVuQi3vPNmxWsKMJyTPTRZXC2xiyCWKmOGapvJQj20qz0nJe2YqTdUTWDIiKK5dgHMrh1oLwoJTqLjzmxfFpMmRglmelZA6Kg207w0jW5m3ICM/HemUbFMVmYTJjkrHBSD+RzUi2BdlDv4eZPZstr+C9MrQFUAmwIZ0c9g8dIQRnZBsUdzvpNpuW8Mhvj07l6tFiu/+CnR02h1T63n8Z5qtHbOccC84oOemClfX03cFgQAVpKmaeoppk/TqRomkkUlWvF9XEqiobtDkSb5rCe0aebPtT6AmgyE2Mqzjtcm1NUGQiHPOkPTK9ytcVp2/wcKUm9Uxw2yaXDRjSCAVMYNU6SQL93z27KMVpGj+HD4RxwMGr43juNbDDoViy5TYC8nteMmq90DG/FSAr9y35Zw+4tPy2whRH7ntCFes5CBfxnwu42GQ4pVrzzp4LWrT2VGpNsgFWrSyWSTtVv7Ub7WIQZzO4s2Gpe3zV5WmBOeKzPd07ZnguNPLtwYTLdhPMVbm0j4RgKti5qvc7Yfq6kG1QoTxIKYHKTpCWmqMdnrKppetqFoQlTW5VfLVaLetnZ99+u7m+vr7oY/EQZaXjq2Vl80d/mZM3Rrpcl+R+xVkLtrYz+N2zfu9gePuQDzNtcBV/1tFcKbSTN3AqhtHa2VRtjtefrZKjw8O9IM0WYJ6tgC5XySnFLe8CNUeAMcKQ9UR0IDZmMIE97J/uhIp5+oGim8Xa8aSIwHoTcI9OwxLC61ev0tfPaQL+YuX+YKUgHBACmkkoIPvpVw8Ohg/+pB7x+6JPD0OpYJXqWJW/4QoeZD6w5Nkffncx6ZKp9sGzo8FB/6IePus+o+fkGbS7J3cX3WfqNT/LB/x+eJc8o8rK/BHXddkCn2sekLaY/23UlOpQiiSUnhhel4GQUpniL/SmWcVaTLwX59BNuSAl4X+jl2jhUYAAxfLDwyBklD/qoaf9LvVXPowIuTtLGlkWRcor4cvJI9paTOmYxq36GFbQgXOwMEd2PxamhAEQPrUUVk4ClT2HVOQECW+ySFPrNGyYKpU3Drcm8VNz2vSfv+4ppIue7RDPDWjIGu8dyI1WpUdtq5zt5JjAAhRZJISsEo6A89JM9Bq5MDotgwq3yjdr0F22Bs8UKP8QOcKKe6B8QL1hAzvHyyFtg2jLOGSK/E0K2wzZAW4+kXSsMjaJ55flZSGjYcMV+uVsCWselVJ1f5ktRTV4En6AJbGSgwmMaj0NyYtPut/6Q+B767E0R5XJhcwmmZlDEjT/7Uuyl+dWWUNZio8+t8istAJMJRe//2pF0B5ZDnc7Jz9kpz0y6iVtEJRzfrcdLztMjuE6nrMD7mpP4z0Muc7u8MPqqrQvPEgU+OwPnNXB2TIVQU9r3qzKczcSIDbTIFXJeDg7iG8KluH5Z9HgNqaY+pQzYnE6HjpNiuh6BcHLJXJnGYjvLProyXvuzVsXf4Heuepwf1yRaCJfk8aaDowc/ZmqaH/y+ARizgHb8a4UOUuCDrZeli0/XsJyb7I/uk4A1hefG1mfQizQ9qJtun82cJC8SLcsJS/RFrK+gWkTnxcz0XTS6b5okb1I2SZRJ2IuShGRQ0IpvNcykzBc3yTFKKMqkCGSak4jf3DJc/qnp1MDndMoWe4v9whf8VxHCRWbVsHYk8YIngUoGqjnDu6ZT/NAIIw36GfOsATKK/aMkzTJjKpWI7pb0rdvleNpIt/TWKshQHD3tpYoq0/NCitx/E4UaH7ZBhSrTbl40Mku0MA8KynWgSggkxZRP+LtJXV/Q+UolWToox/BoayFtpOvHYnBOaZUOpBiEg5u93NVtezjcsucQkahGS2D0aku7UTybZ9jdfkti1BWChKBMRXSW9cOnRKJeHzmMiaxUujxugibo8hZufMiYhycWgbTkhnZcD2w0FE2KnAs4eQYXIIP1VaZEalheS+kFA4myxNH366Q6JOmQ0VaUNIc0B7PvcDEMDUtTjGS9kwuNBAAziSgB1YVI1hi1lBMDHNz6knPeuN0AFCZon35PWFo8FKGd5ulZyIYfcqkBKbCT9Dng4hzeGKh4rT4mMSpaCpNY6F05Cfqp49L18LSwkSBZ1om5tMNmQ4YovXOhevA6e5QRGptQiERUnGTwk80UQQT/FuU7vNDz4uqbcJjKYHShuUeUAfDbbAD6hkCbBl/dnoPBjrI7nS0S0T/0YpvnyZj/4nPj7ZBay+5A7DtrQdIl1hEV3AZvLyyK/Yi/YmCjd0vJTsphiuGXOk4gJXsEGtykiF9i+NcFzrRyDRqiI4u+dxpjok9R6KQryjdmsnGJp6t9Gtb3BVkLjdaig2DdNcrIX8ZBuYYkRUvM1+ltVh+aABrWmrl167jC6N88Vj3TeplUYYcs3W3+cBlfe3X2N5X5LcnSwli2KYNBbZh25gibkR/vLWJTXox4Vxh2z6s5DIKN+mmYfQIJl3ENj1CLwqng7iPqUNro5AhgU3YKsdA8OTOlkk+s1vd2GUrVVvbHvB9zcD+rbCvJOGXsg8MbzFWIyZ4y+9oN9D3Pgy7ZU2q5oSjk4NtmX7taDFMgK51AgmXkGJ6GBe90mvTonD99i0uHFrQ4A8uNyTf9+Ao9Jrl3EA+t4BMt9GW+3FXYbHqrOwM5qdwYnfZoLQIeF0T6Qg0eT7XxO+DdaNOdsLfauqXsHfMm0bHuqHO7pirskJOymnbBxue1TDgbEGoP56BpbJ/kBKvVOmiTsUSMGxKHKecVR1ekaTiboKjuIsZszs5GZrgQijasALWdLZ+1bRH0RnX2hGviwmJ+zJfqVCk5kJ2FvxSTMTwJqf2ClvrJcp3oz9I0P4eQcmoHERvsX9JNSpXHbBgFZ0JaQuCMsORsqR4p6Vwu8EDURAVjrp31rGnmUTvt20gUEb+GV7BfKcA6xI4ScV0/enTzYFJYGRaE0pIMWaRl83W1WxyOlTE2uDQIM9YPxflj8C9p3789PZHUxyeP0hRi3b79I+qwuRIyd+dlV5KKe006wRPNnSbW5a7Fd5th1qGdHThoMrGog0FVur5lHQ8BrklMTCl5CThRGdCFFo2Jds4ZUkpuxuiwMVMYCtRwHljIFn36r1siYuwjnsm5iEM8fMBImxrcpmveN1+dLe4w94+5Dxgwpk+dwM5Ixz8zUND1WzUzQ0ExTHavPYVN46qWaew0hrnXicUSOZPdksHbTrUszAOq6LtKd5pV8B5KMxqOvObBZvotNrnHtn1B8p2J9Jq+0XUeynpCAYu0hfb6TMUy7byi7H41CwodCKyjDHP7puDzFXJKhWrP02tU6Ifg93uC3Y/LayNiNTGOirQ3epatRviY2B7bGSyUl9pZ5MuO/4q0mIbFs9Dz52fjB5FuTasSTJ6PVOS/aaAVkcJdFjBLTHzMgQzfJXyU5ZpJOPUVXh3Z2Jg/vY5Fj8RILqDM7+nwRzubuqGBdLpVZymuVSwXfxEAWsQnMk9UPWHfuQh8xpdbePBcUr/E7ZvGL7l6VfX+Q+vjnKpY+BsAhbNcESrctfT4+gsmdYUD7fBpA+xPUXTDHpd1c3eHuVHfpaIcB1FR9l7VJK/WsprMSL1RAfU9N4khqmYARJsgIgIPxUEKYZbkQrvmmJxlYjbovTeqvh2ld8u46eZC4ccMJHEMx2JiewvfdkTu+zQmlwOC2kpywpwSzXicVRf+HX08x5/kNlaOrsmTa+za5oEU4+5kyKUB6t1HrTbL9zhbLVfuB+Ge9guXDo2rcLWDuOTp0EOZQntII7uP1+9gItZ03Vo9jEt4aYmnPpU2673i5XBgd1gygWcaFYbvt8L/xa0HvCm2lB+NAQkw+mG7tCiP19T5OJ6NqEfAM1/Fnkt/tK/sQX8OuMPOGzm9Lfgp0w8XC/EvxIWJU0Vf6mQib0z8XeW6oD6eJVPLjmjzH5dfIWEqEX5fvMKN4fBqR+hk6+iyzVFXCKAcZRv5nSGpt8b3EpRQxjFD9aY04+8Qvxk7XSNVkgNqX8D0W1eP9sQ94Qu5BlqwfPwRnxDhuMpkkfgz6voPbJy79GtbA2KV8Wne42frDSoPw3xlG/yyUZIoIQ6ZzJGrCMFnGJvQ2jmeHFZ4qjt9GcO30siUR1RrCrUOpc5BUPiV51vWBkh3Nq6e/QhryEK3asDdO8dEONoTkw7OyzzBlNRRWc6sZvIc+X04Dv2+Iy+WyMoEeOKH3AwWG++++47TGk4Dcxhq8lQCyT5gHRy1T4dmS/u04tpteZbUy6uv9lUsM6O17PFut7geI8I1PUSXGhDKc7rmgpMCvPb7sa6BF9D6CMdxTdTuGATiAliMOncQIVB3IoGGEdcCIe7O6GzX+zZixtKE1hNsBJWC/oX9xusaFJc5/n7zRyy3xUHfW9KWjHdjZoz5nPX6U92udiMcshRWA5TAEY8ci4jhTd81d9GZDrBOoOgonys9+gXpzKo9l//c8Q7bMQfciWkZIQUqJwxs4OxvBJX+IHv9eBJKAoCjc3FCGv9YtS/Pew9OoR34Gi9akdOaWvkbQG5hd/WbDBWwq6QTowtGCeNPTjWGjbp7iyqxk2QTf9qBnm8BSTVaUIK+lUzrKMmrGXGGSeZuHQJnjCOBWD6XtVtyImSnAly5sTxxiqc3srmaVleVMDQF40n+XznfPbKEKg4fpz+pt7Mgq3z1223shhHdAm96YJu6WolKiLMjVPRkcdys1h8K0ve2d7vXGNbWTpZQZ7mk6kwICeyG06sQfQNSd3tEQfNFWXr14WVJLDsdmTKCamY2VZnrt/abqwzLnN+TLhbztNFOP4n7Kon1JvwlCO5nV6w30li4BMMvqgVXnEl6dG1cjEWN4TFLLdjHgFIn25F4mq1n6rkDbHduUlqb1q4i0iWE9K/zFaSlbzfUkZ7TZFINmcms2UBpvateGw7ltuOu7bmNGHLcnrBbmYgPuWOKHjHqpFZwMDukdvir2/e/Ugphc3ru2HIN/DeXjmO+w31CuoAm5rCQ8Z2kDnHlXF2R7TYvRYe29yrPqttCB4/EkQGPBXPyIewTDkhFapbSUfBO2vCVhst3OOUAe0m2Paa8RM+Wk0WZK8oKbMHNUXDk60/8k/3UAqI8Wt6TTVEfeRcXyItAcwf8d8WxTi39OTDz/ICJISChxbOHObQNeSs5lZXGTVlKum9aitqN9p2h2FXdWdbFM3vBwC+bJyhS4rNxuWOspZ784TLxo2hbgPthvVJrHimo1KfvtbdHUmVFQf0c2/SECtXIrE98m/P/8lSWUvnVgVvqJmzuhsJLkb6JMs9GGVI2VaS6sdihJbDLl+BSKmRKLLpTDLOVL1JU8eFV9zRxcnDoYJFx2OhKE5chmlRI/avYYSNNB9Teh1Ude9cERnFcWhESuWav0tXFFBBQrxrbjk92Wu59VwXM+Rog+oeexinXVmPGZxy6VnhJuuZ/M2huvb9loen1u7Dr3viq9xt2NRGzXFYp76L+hcWp77Hy05Hs4kVRboCaCglCd1OUPb0L+WTSMCkTlNhjR2eQXGRT8JRla/4Ui1Sau/yQzqeTJMjqJq5S6h0HJKwjmAPTSSQb745VrBlBAGGk2EnFiYPIqd4cgwZQ9TpurKHzNlSQaPgGBRldf38jeh0/a/KSxNvNfpNxF0xb7oOFdT/fNweKtzwHDJQNfF1U4WJ/h/IXvTUiwfyhTqXIJ9YRpF6fLfaTZv5WN6CpH1PkUONTyOumBMQlt0dWMwiK6G/ZXwxuYtFY0ajqSePfYePTGOspEN5ZY/ti+Hf3wMnh15EjnmA07qsevYcuu+uxTTBOje0CbBgyetY4U1dCOo8BWM2WAW0j6zCBqpBdSfvV41VZ1Yy6//1ym1CkPQByYhtgXCtRYigKEN/fpsbt3RboatM6HCsrkrT52P4c3CB2j8zW/kS+DIXlXqoRk6cFd28AauIrKre2GptriMuAyHRRvj6yvID/qiz/W3lf3YnBRCc8KDErLA9y94ocE7cs+7XudeoU6rr3jugr0JsPbi23D8g0NcxrKFAma2xvL2d5ThSd2uxRhTu1tKN6FqrtBMu7dwEHghzDgbvcPyX9aXd6ckt1OLKZIf5KB2DITKLFv6ZvmviERUXDviwaHcxPyIpCmef50sUfK+++ryR+ls6kzWSoFIaetykEqggs977ddQNIagRzLdMwWRFGa6VfdxWC1eDeLVYcyt9fFHTCgWn8vyxpQqrHFDFUsrsqtII4Eb1ZlB3EIpzPYRJ02l7AqmErqz9LUZrKTC1Df+gOf76gohh61ygMQ7dVcH36+yBpFqUZnoaEb7F49+XK7gDPZ/xiIwfJotxt9s4vNopjhkIhy191pqwvFgbHjG2yymBWrNI/AV67HGB//d63Gl4OzUjgoOeWbbqs+ZlUHygsXVlPukf2vRbMi5VDU+6Fldq+yb5+wFs+tj6aleRhshl2A/86WpfQ+/2V0UKOYPZtHffWtKf1vfG5HRK7gDkRAtgWhCixS1FOdM5rZztkz0i6d717tmK3N5yczWL1U5wtH5zk8Exs1oNj+k+zYYdLJq4hMt5+Nne0/kkbvUpYZw9cTUwQLYTXouHdkBFc7fdoQAUgnjI6htrDvp6GH96ylubyBg3uwldi2QtHlJPXeKiDip8bveg/RoFXO8HUa5YTJL4Z5O0iE6F1uZitjAG7excX3CvMnKbw7sDrimDXcc5FRZsMeStd9RxD7fcUacCUGp1RAzFpGy7ok5dD+SwRwtaSwzOPjBdebbILaiBYIRtAGfkKlCvlOlRXKHngnK8+/lat1Z9479jU2wqSyz9oiOI68GXz/YRIiC4EaMOvA4cbwx8741VtimGsHbSf+ngFGIOjFXog1XTZQ2sXLVf2PpPf33RnY/+O6XdtAK05d5qtI33iVTYepgVWsz3SF+IzDaS+VpZO4W5Rao2X1B3/gbDPrnUkP3o5S9//unV2x/Tt+9+fZ1Smry3iNJD1ObUFApISFsiDHZ4sX+p2AIvmsBzNDcxACvqCzvBK19WS4q5n497O8TeLld26yLeXf2vkemqpOk9paM7kZHPbK4RoFyIgFs+vbGXmnjmP9JHjb3l2E1NnCLFGY33XfoBzxwn0ccVOZ4JDzv541r8sBT1uF5jqdGxLqZvTiJxNX2wJzJFAFERULy5ltjaSqEd8W5yNzicm5/WRe+XmIYpRxCdkx/3scVZyYl7Mj0/td7QKE6R8uGJ91IZZM5FpgCr+fWITCY4cK+mHmbCtVL6F5xTwq813GUGyPlMicjk36+RSu5IZOSyZjB0aFXxkXxOiRzQ6a5SE+9myyCBDJsq5UI1r0UoelZTIVjJGBaMkqo3wphDF6ootbCYyyCjKwy1erIGhARqmFSx1H7ImRSia8rNoJCFnOixkGzZ5HWIRJwV1IWz4Kaqlhq2Nrq3FdYnF5yEh3V/KPTe22GGLkl1V6aFNwxLNYwHfD23r1kgFyKTw68hmJkLSdyLMKX86XHjLz75j3ZOfj3Pj715/hsms8hfp6e09KD5jDktAAVmtnLK2Ta15wi3SzOHr4ddZjh1QsBbivwfKEBDn/EZ4mhfiI5nVgPWDaPmd0kSbSBwHwYnciBp6A1jtRXYaG9gwuLYCu6+YMRfMgHS/OJoXBJ4+bUwK/D1R3LQkGIcjmHpCEvuEJVBJVUGfVQ/rbpCXbW4TBeTSa1IiuckIQI/EEXJ3IjXHOhbiteH2J6ohDSwAoX84Oi4xwGGXQ/qyIE6CkMdGaijbVBhsV8RmhbOB1ZT0ohTFXSFiA5fVs1YfXJhU1ysV3IULskN4I4cdghuOxh6qdQkh7dWo3+dFonA3DHGgI3BqhMe4411cktebqhIY2y54gWL81risqMtZWVCFlbxLGfw8OU7q9ml1rjBsW3Jt4fIwwPu9UzJB7m85Ove5Yxt5HhzN4pPof2ho7nxv53NY2RP9mfzJ1+OzQssNJsXvf0cNi+39FWqCbbXtOTSJAWIhls2g15zLmu/H4vF2Z3ZyuIC826nbOJQqlVCCbcQmnODUJM9p3PDFqHFo/UgtrIYxRoYkoXisMWXI9X1Ap6HMiH2DOc8sXZalAdanhR8guRC9caxTsrFKBlRaq1XVJFve7ZOwsGatRLOG/e8LROdCjVL3Z7vlD8IXUzAw0NuSOSsy2WEk1HHDlxWWVF7bWlRoSRc1W6U7WI6rSk0OkmlIqdLmpy8JHZIilRXp+Ng5yp05FXtLF12nRurVmGHDYZlspFSyLbVvZ6lwtuSBhTbiuiB75IjvQm4+WbyIO4UC8uY/U5eHuUUIY1LXP+zkofa7gEiD6RMMfvFFC0MtT5rK2Rir/VtDXQF2f+3zgVBJ4Ghzp1Lq1d7Ezs9HtqszPEoJqN8LDKCuJ6OWgvIaSu1DU7q4tVXblZvVhIHFwCPEYV9B3hAsyE751nTsBxq1k4FRCvo2GRcEJNOOD3mLEmAtc4tXt6W4qBrzS6TX52fmTE7bwZtl6o2HBtaDeX4+apGtBhrexzfAdstwIUetPgHCdTblgDWTKGhNrSIcFMJi25YpbvucH0zNSau6OWXHG56Hsc6F7fcRHaNuJfLNDTQoUNcY9jCw7ttdEksN+KWO8ou+jJww371hcc50EVvhJu2CVRrxkn1omDIUy9qRi15DXzOrFBo8wyQ72RoWmhSyDnomeDtbdcyd5s9t+GSdObkO8jlrbqUkXkEKBzR4fQtYfm9oQxXllDfzpnIDbyR2jtUwUs2gX1WYNGHxWp1c9bQlTWyUZiU3ZxNQ1QOqbhNKpftU16kXGA4oVy8O3ThWBXHtCq0JIEfZCJTVPYgNv1DvPh/L0+0nmUmP/Sk6Ygw2GMZtacz0XP2XmlNjIRmJmlrkol26/5nZom+r3OAdPZf8bGVzlK4Q5cdJORaaK9077TSWznEmRnRLTVVzuWtmZn3SkDRNL9azgdy5NSW45xT7WF1eb1ls3QgiIS09puO7eIjc3eHHJJas8Q5TkmyFEUFe9xQfqH1Y4zJTqtb8vPJg4c4VZ75uXTsRjlq9fTw0F0DI2zq7zvOqQTYydPJHtZBTiemaxRElsOh5ZkEljNxPx9Z+Wss9PZ1a29Q4Mwi4Jbin/JdjhGNnDHct+3JJ8PJY0S396jJnhFWU3vkwzT17Jb2rXiP7DB7p0RXK8v3QraWlb00nTmpY3rsLM7Gm6Lh6kalxK31gZBnp4CfZ5k8IhUyCJFy51lrLZ+IAILUMQdQKBXltBfRhUz07xP+9+kpB1Lji1dLpMxGYOxslVIV2tDob0o1xcMTenh6Sg9IHM8/LSAt4cEK9V1St9NFF7WQBG4NhXzyyRhMqrpLJm+Vyx+6atCue4NELcbEzCb9NNjh4LiHWGGydjd6GloSOymz87zlutH5Era5ssJI2DKLtydhW/oaQSWpcpEJvbX2Vcjs8AC2QgVFD868oG+k+CIZiQPgSj9sIxgfrmCftRc0ahiSXpDtpeSt13WgsI69Z4jOVvpU4taXwjdPvbL92e3zMVVTN2CKqFN9EApWbR6gCYIdJ65LRNYROwzL85ojSOy5Ctn72ngjRTRLVtsABXa2JFY3abOm9wB61oJ9p5B+VHLb3XAbImAiLynRsiD6SK2GINw1U0LQgQQjjENtj4IzVqwzdYa3mRlCTRU7m8Otb12SUNm8pVro+P6B9Jn++LaEgg1YsYNHusCxCCF1IQuTsT2lxlZFWnZSZhdVw+U2RkIWWIgm6brmXRHJhieZzVTEwZ6jKul2QN9q4Ll3+6fczyaVLOVdNoEMNJzmnt7V66m4gMJoG7phuprRb6Gu0ECau1f3oelSGD7SqzUYmmTjDWsfMirIGGliPg11F9b1LPxtehR+3xgte5hskySNYmOw9Bi05fW47QQHw4iCWoFkK3G3kNmQeI0NfARd8SSoHdpO48kCHlF5yo5mjTizzyHJnZLPmtuHo5WxZraXwbUleUKvs0eWg0ChllQErRqa3atpzxVFjd9jSe2zrJxxF1lLUs4oMGmlgZd1oDEqIu2FKhDC6TPX6B7rdNda3bZef9OaDSQvEIOu0hZU2I+k1qROWBWGe5dkvoW39MjSDt3EpPNdcak+BpByNeSU85qFI3E/Eb7IK72QHgDjCpxUCT+dzd/o+CyS2cSqSQGCaiDPGDGMfKV8aAWScNiuOJUutSRFZvPFK9ifv8e/ZKsj71fhvADNEAiQLt5b/m7T9UzcCwXIVjPfgu50d1qNOFLoAibkZV0tMZMAEFbdVSxvGZzm4cq05aX8Wfk1eDUdQnJ0hvicaIRUkg7LNC4ygYZvzhPeKkbzaHRS4Xvv6AKR4Id1eV2R1mHScl8e3Vu210163mdOwxv+pLwteu5VgAj4tl/9/dXriK8+dMx8itB6ZmiikjXWTuEuM5zKNAZuRgOHspjo5osHVJaRAQqunVhiMlC33dGxyL5jRX93NSCN+1UaHdO/RZ90VzxAXkV7VumZ6s0qazL6a5eqj+sPiTexIdebIvgeOxCxxijFM7l0iCTPBxCh467LP2RLtIRxClg5ufOt+L1gWYWVm+7IQbFRZU8sm94CAoXmDQ7bEG6WtsfBxZRvbbDC2K06ilsEuIC5TliFT+lpqnMONM8asmxwVJFQjFOJ3W88DR5C06KxEEfvJgr8/gsi4FPTO9z3/wm7LIQTsdHxNU8aFfrUn8DbVe58fb8ytUv5YOFeNC4KhQshVq7O4bHvoxXb3tMQF4ppzhrg22YuIxItKHlq9JYUPW/FoRM+Ho48q7MjxR8OT518CpVIm7BejenMtxrDCeradiKLxdaIj7dwPRCGF1ZXc25xCCqQ6cjtUbzsyasWOdsIv+FxRc2efKSEcjVf8mfvvF01vHfNe1rlufnMP6BZcTfq0I0bu5BQZymCFsHGyKvLF3DNnQkpHQOoFhWLbOaHe8RGJPKKSCMXaONcZdCQkYFk23P5fEuQkMwysgVBAzD5r71AdsMwVa3UJHqQmnoXqu2HZ4/Eim7ahNcuW96YNag3dpKMkmQ/UySwA/DAwymUpj2JsZwP0p5tUoCk0bYRGGNk5ilfsC0C0LwD0PP1ajGniayPhD0rcM5cU9vzr7bp8a0quD0d1oF8hjAfSHj2dTexT0f41iJJcl6wkZwDgiS/QloilKck1sgvzcoxIXRO+tFrE3VVMVSKRYoggfME9dsIjCtLmP3Yvz/rzpZSRfsQ41iC5JSM9APrzRJjyXpLgqy6Gx4p5muumcgL4RWvGeiRcA9zS5H7hrzLbjmTSDq6ERYM/kIjn9C1ffIO7ljr66REwm3xxal3ztnWw5U7Y/VRoRUWj9VXwaKbFWyebbFqVebezNnRDDstdf4vBVKJecLJAAA=",
    ),
    "cfpb_seed_source_eda_v051.py": (
        "0a21491e1dd6a1f7a10d3e2a489bb5293189febe167c1d1109dc4466bf9e4caf",
        "H4sIAAAAAAAEANVZbW/jNhL+7l/B6lpU7jpKctcFFgZ8i9wmuSuu211ks1ccEpegJdpmK5FakorjffnvN0NSsmRLSfbDAXdB0ZXJmWeG88YhGUXRK6U1Ty3PyKvLt38jhvPsyKhKp5xcnJ+Ru5PnyWkyGl2vhSGFyqqck1Jzw/UdN8SuOVlq9ZFLJCSiKHNecGmZFUoSJjOi7rjWIgNaJfMtMowWfM3uBIgg6ZrJFUheuAkvi2Q8FQbZQS2lsynMWn6UC8s1y8mSibzSnABnlgu5mowWqpIZgGiesdTJLTgzQGMmxIqCf1SSHxm25KC+tGtQYqVVVSIvAeVQFHCrypaVNZMRKo3KpGgXUyqZASVhVSbs8ZIzC8BHmq+EsXpLloLnmUlGURSNRmCIglC6rJCGUrSG0hasIJU3iBmN6jFj68/fjZL1d8Hsuv7WvP7aMC1BB+MFlECTi0WN/hZZ3ITd4prq8TO5baTJqii3IJPIsh4qYZkwAP+VWdA8qVnTZbmgGAfUxwEFw1J0L1DTBTM80A/SBZwfCPkTkeoDm5LLH09OJ/C/v4xGozRnxmBsvVJyKVaxg0ya3+PpiMAfWBShUjdWaR9QG2HXhJEMjC9kaiFUlqzKbXAe0UrZxLkCEWASvFEqY6mQwlIaG54vx+Tor+QXcLkXg3+emyI33TBDTVWWuYCYmBHkSFrzBJIAvOkAGn6g5zoeJ11h42ZeLB3PkJidIg5sX+KMxB2ChqjU6ndIXEd1QHFMohDRUd9cxiyjpSg5pBDvpehzbi9hcPppd3IMjkYHVHKjWUkxhWlI4djyezslkD/OF7aCmnEDvyY4NG+8/85qkVrIVg9B3l9fHr1wtYAEIEM2awHlSHPLBGYI5E++VLoA16EMFwqIdsfyiqM3rXbCvWvglyhL52ZHkLiB4LfgM80TyMl0Hevot0V88/1tNB8nP9yefhtNGv4JWeZsZWZA+268cyZoVWnpoSckAjTK0OEpW+TB5FBBdvQututUT1KUSuuf8bgbJA2ZcRV3KXIwSBxBpVUaNHu3hQp8/6snGn8N5zmH2p66ZOtlL5k2zmJQwJLgBsphjXFtjYOwh1oujWUy5bHnnjgnmr0l9ZjLkVG2BCnUsUb7xvWACewYKuNxVNnl0YtoDOw+ZkAdCmMvPB+/T3lpSdyxTu+SB50oJHyJjHKTspJDenyoOCysg+/hL9CgWDCdZXeABTeGrepgdJPjJFcbLCBty9WiiBcVESEbXgCGBQpc8+Hsg1Z9UP8H6NvpS41bYGfN7702584PF90VD0CiX7yHO0j/QrKnIXSUcpNRKDosyyjmOb0TfGPipWYFn8JGl5xD2bvEX67ytAe8NL8V9dQs8FfP6MhrZ3ALmgXmPtnes9i5YDmFXqAy0A/NyKdmiYOemRyS9Dmjh6xl4MdAvPFaVP3Z5wm+tFZ9E3VwHBn164zmsMBeKm+AaJ5gZdjtbXvmcePjYVldgz1BGv8QP27ujlQXdR60FVgfKuC0Wxp6wa8JryZUHEuSqnIbsj5lUkEOuUCrlyGZxrbnjtNmFpYBhXdbQq3DeitXkWfP+f0gc5gbYq0ZoBXXNIVW2jpbNiJxV0xyLqG/CfxC2i7rBrr0FmsQCBtnubN4zopFxnwOTwni+XT59c3VOb26SJYC+tHcbSOxIxqPvZBhsQbOGeg0L5queaWxL0z39H9AjYLdx6cTpw1s3rUKOrr5Lfnm5a2cP4tfTm/gc/7s87djv+M3yj2mXXMaGTSqGwdptzdX2PxeX5zfzqNhQGjZOC1zlvK1yjM+7Kwd7vnZ9cWDmKxwpvs61LPXb97/ct2LG1xPMy5VISSzsFXNeuMk0dxJjU8mcCpJJJMD5jMUmus6iClCOOXi1u4wZPDjPn2C26DxySWLT5ITL/fABE+T0VA8e9RPPaTD5m/VoXC8fYJNAfkAaiAm23RwjsDjGgx/hWV7tBpyacfYSZpDjw1fE3Ja274xR54/TbFDZ/13FIIDGtcplCnqzp89hcUXMbpH2CTGMldsL+XwfAK1ij2eax778s3Va/qPi7Pziyuok8OpnHOLO3XB9B9PRP754voaQF+fXf3zUewVJBHAF08E/vvZzxSwXz+ICkd7PLcZWum8F9FPt+IxWltbmpfT4+PPm83mNoGSjEafXcJZEDpDzVf8fnatK97TPEAswIYNAVJWMrUVa8JqQCrsAt+8TCZH808/Tr5EbfAubt0OAEwmXO7fcb2lZg2R2+lK2htsktv49OTkSUi5gr16EGhl4+f05HGoQhk4S+8CtYN4EOcIe5I8fxQV4vhOAKyvEN2mb9dsPVp2HGkQevp8uP1aVCKHDrEqIMrFA51XJlLrbxTaU+FqIVVFCatcCFxL04sxY8RKtmtfrTMUFAOUM0f3lBraCr6nnukbEn8eb0g6Z7kIFFS63afjXzj2zaJXSkK8WLwGsYq85Vqo7AyawO3xT1AY72FF7tq2vpvdQAkkGajd3JHuAYO2fKX0dvbecF2flhuK3WG1cUdzBNp3U8fi4e6lnoSCiHYN1FsXQMOTUNElDOzskqq8KqSZfepaquCZYJIeODGa9sw9FJT496Xl0FKVVe4NWEdOrtKbEBu7WcpzsRJ40TOf7y843EL7FDb7W1sDUYekI5/txkOHoXnKob3P/KELMi/JLFawJbozjr7799F3RTTeuSlxl96LbRwUgJIWvdUqq1IoIy0yIz7y1lVEbfFIq42JOuOG410nhFYg7+Rts95O6rpbdGoYXj2F6O7J4Um49Z3uLolH/1t57daxH/DdxXXUqVfkuUMdxVs0h5OsuI2b6urGouYWsiZuXTzv6sYOKXwdJEgnSaLehHh05V96st5rfrOn9XynSTsYPHGIhHBnX5+faf2W4nXOYJuGmJLpNpzcpy2vn8ntnHx2NgBB+M9kPzC6xPP63B2ea2qHDepwIH7cAUggDeMeK06cMl3ax88wresfCA/gwVxsO6Td6BoCGBAC8qg12Bz8HWbrAqeSwkFlKq3wPa49ZwXXbo5jRyekOzRTtcAnPd8WtYjdNYkvaeEmZeqvhzT/UAnQtE0MDbza4J2r4Uh3g28DWGdcTuCVwwQX58IKe5rufdKB1R46CwyZ7uAtsGOtFsj/t61Ca+RNFvJqowVsC+HRJ0YFoKy+DU887/CnSxV8L/RZUTApltzA+ROGmtTogRl3yIESnyxh32OZiTsgUH2Yv/2MIYUUPpzO6kv5LshNFF5enTcj9/Ab7VGk/mEa82ff550bvvAibCiDTMajRgb2xEa9Ze1dUIW3YSoMrY+McCRrbvXsGpAwXMwhSL1x18/HiFE3TxQfmDssXw6NHIzrDNTAOmNmVVHujDkhUFdQSWZSIerjDe610s7+PMbprnUP99+O3NF/AAKNRcXnHwAA",
    ),
}
package_dir = PROJECT_ROOT / "src/findisputeeval"
package_dir.mkdir(parents=True, exist_ok=True)
(package_dir / "__init__.py").write_text(
    '"""FinDisputeEval reusable pipeline package."""\n',
    encoding="utf-8",
)
for filename, (expected_hash, payload) in MODULES.items():
    module_bytes = gzip.decompress(base64.b64decode(payload))
    actual_hash = hashlib.sha256(module_bytes).hexdigest()
    assert actual_hash == expected_hash, filename
    (package_dir / filename).write_bytes(module_bytes)

requirements_dir = PROJECT_ROOT / "configs/environments"
requirements_dir.mkdir(parents=True, exist_ok=True)
requirements_path = (
    requirements_dir / "requirements_cfpb_seed_source_eda_v051_colab.txt"
)
requirements_path.write_text(
    """pandas>=2.2,<3
numpy>=1.26,<3
pyarrow>=16,<22
pandera>=0.22,<1
pydantic>=2,<3
rapidfuzz>=3.9,<4
datasketch>=1.6,<2
spacy>=3.8,<4
fasttext-wheel>=0.9.2,<1
presidio-analyzer>=2.2,<3
scikit-learn>=1.5,<2
matplotlib>=3.9,<4
seaborn>=0.13,<1
""",
    encoding="utf-8",
)
print(f"Bootstrapped v05.1 pipeline: {package_dir}")

In [ ]:
import subprocess
import urllib.request

if INSTALL_DEPENDENCIES:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)]
    )
    model_check = subprocess.run(
        [sys.executable, "-c", "import spacy; spacy.load('en_core_web_sm')"],
        capture_output=True,
        text=True,
        check=False,
    )
    if model_check.returncode != 0:
        subprocess.check_call(
            [sys.executable, "-m", "spacy", "download", "en_core_web_sm"]
        )

lid_model = PROJECT_ROOT / "temp/models/lid.176.ftz"
if RUN_LANGUAGE_ID and not lid_model.exists():
    lid_model.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
        lid_model,
    )
print("Runtime dependencies and models are ready.")

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import json
import pandas as pd
from findisputeeval.cfpb_seed_source_eda_v051 import (
    EDAConfig, PipelineState, add_dependency_features, add_duplicate_features,
    add_lexical_features, add_pii_regex_features, add_quality_features,
    add_text_views, build_audit_samples, build_dual_frames, build_summaries,
    default_feature_registry, load_sources, run_language_id,
    run_lexical_statistics, run_minhash_candidates, run_presidio_on_sample,
    sha256_file, utc_now, write_outputs,
)

config = EDAConfig(
    project_root=PROJECT_ROOT,
    output_root=PERSISTENT_OUTPUT_ROOT,
    run_id=RUN_ID,
    run_lid=RUN_LANGUAGE_ID,
    run_presidio_sample=RUN_PRESIDIO_SAMPLE,
    run_minhash=RUN_MINHASH,
    run_dependency_parser=RUN_DEPENDENCY_PARSER,
)
state = PipelineState(config=config)
config

## 1. Frozen-source validation and dual-frame construction

In [ ]:
sources, inventory = load_sources(config)
raw_union, universe, conflicts = build_dual_frames(sources)
state.sources = sources
state.source_inventory = inventory
state.raw_union = raw_union
state.universe = universe
state.source_conflicts = conflicts
state.stage_log["source_validation"] = {
    "completed_utc": utc_now(),
    "raw_rows": len(raw_union),
    "unique_ids": len(universe),
    "conflict_ids": len(conflicts),
}
assert len(universe) == 205_589
assert int(universe["population_eligible"].sum()) == 197_489
assert int((~universe["population_eligible"]).sum()) == 8_100
display(inventory)
display(
    universe.groupby(
        ["sampling_frame", "population_eligible"], dropna=False
    ).size().rename("rows").reset_index()
)
display(conflicts.head(20))

## 2. Corrected text views and bounded quality diagnostics

In [ ]:
universe = add_text_views(state.universe)
universe = add_quality_features(universe)
state.universe = universe
state.stage_log["text_and_quality"] = {
    "completed_utc": utc_now(),
    "byte_literal_status": (
        universe["byte_literal_status"].value_counts(dropna=False).to_dict()
    ),
    "byte_literal_parse_failures": int(
        universe["byte_literal_parse_failed"].sum()
    ),
}
display(
    universe.groupby("sampling_frame")[
        [
            "char_count",
            "word_count",
            "redactions_per_lexical_word",
            "redaction_placeholder_proportion",
        ]
    ].describe()
)
display(
    universe["byte_literal_status"].value_counts(dropna=False)
    .rename("rows").reset_index()
)

## 3. Privacy risk and language identification

In [ ]:
universe = add_pii_regex_features(state.universe)
if config.run_lid:
    universe = run_language_id(universe, config.language_model_path)
if config.run_presidio_sample:
    state.audits["pii_presidio_audit"] = run_presidio_on_sample(
        universe, config.presidio_sample_size, config.random_seed
    )
state.universe = universe
state.stage_log["privacy_and_language"] = {
    "completed_utc": utc_now(),
    "regex_risk_rows": int(universe["pii_regex_risk"].sum()),
    "language_id_enabled": config.run_lid,
    "presidio_enabled": config.run_presidio_sample,
}
display(
    universe.groupby(
        ["sampling_frame", "pii_regex_risk"]
    ).size().rename("rows").reset_index()
)
if "lid_status" in universe:
    display(
        universe.groupby(
            ["sampling_frame", "lid_status"]
        ).size().rename("rows").reset_index()
    )

## 4. Exact, template-family, and fuzzy-duplicate candidates

In [ ]:
universe = add_duplicate_features(state.universe, config)
if config.run_minhash:
    state.minhash_candidates = run_minhash_candidates(universe, config)
state.universe = universe
state.stage_log["duplicate_analysis"] = {
    "completed_utc": utc_now(),
    "exact_duplicate_members": int(
        universe["is_exact_duplicate_member"].sum()
    ),
    "template_family_members": int(
        universe["is_template_family_member"].sum()
    ),
    "fuzzy_candidate_pairs": (
        0 if state.minhash_candidates is None
        else len(state.minhash_candidates)
    ),
    "minhash_max_representatives": config.minhash_max_representatives,
}
display(
    universe.groupby("sampling_frame")[
        ["is_exact_duplicate_member", "is_template_family_member"]
    ].mean()
)
if state.minhash_candidates is not None:
    display(state.minhash_candidates.head(20))

## 5. Linguistic candidates and active dependency parsing

In [ ]:
universe = add_lexical_features(state.universe)
dependency_status = {"enabled": False, "backend": None}
if config.run_dependency_parser:
    universe, dependency_status = add_dependency_features(
        universe, config.spacy_model
    )
state.universe = universe
state.stage_log["linguistic_features"] = {
    "completed_utc": utc_now(),
    "dependency_status": dependency_status,
}
state.feature_registry = default_feature_registry(dependency_status)
display(pd.DataFrame(state.feature_registry).T.reset_index(names="feature"))

## 6. Population-only statistics and sensitivity analysis

In [ ]:
state.summaries = build_summaries(state.universe)
state.summaries.update(run_lexical_statistics(state.universe, config))
state.stage_log["statistics"] = {
    "completed_utc": utc_now(),
    "population_only_reference_statistics": True,
}
for name in [
    "frame_summary",
    "quality_summary",
    "prevalence_sensitivity",
    "tfidf_top_terms",
    "register_keyness_exploratory",
    "frequent_ngrams",
]:
    print(f"\n{name}")
    display(state.summaries[name].head(30))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plot_sample = state.universe.sample(
    n=min(30_000, len(state.universe)),
    random_state=config.random_seed,
)
plot_sample = plot_sample.assign(
    log1p_word_count=np.log1p(plot_sample["word_count"])
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(
    data=plot_sample,
    x="sampling_frame",
    y="log1p_word_count",
    showfliers=False,
    ax=axes[0],
)
duplicate_rates = state.universe.groupby(
    "sampling_frame", as_index=False
)[["is_exact_duplicate_member", "is_template_family_member"]].mean().melt(
    id_vars="sampling_frame",
    var_name="diagnostic",
    value_name="rate",
)
sns.barplot(
    data=duplicate_rates,
    x="sampling_frame",
    y="rate",
    hue="diagnostic",
    ax=axes[1],
)
for axis in axes:
    axis.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 7. Audit packs, persistent export, and hash verification

Manual audit fields remain blank. Successful export does not authorize Seed v05 generation.

In [ ]:
generated_audits = build_audit_samples(state.universe, config)
state.audits.update(generated_audits)
state.stage_log["audit_pack"] = {
    "completed_utc": utc_now(),
    "audit_rows": {
        name: len(frame) for name, frame in state.audits.items()
    },
    "manual_review_status": "pending",
}
manifest_path = write_outputs(state)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for filename, metadata in manifest["outputs"].items():
    output_file = config.output_dir / filename
    assert output_file.stat().st_size == metadata["size_bytes"], filename
    assert sha256_file(output_file) == metadata["sha256"], filename
physical_files = [path for path in config.output_dir.iterdir() if path.is_file()]
print(f"Persistent run output: {config.output_dir}")
print(f"Manifest: {manifest_path}")
print(f"Physical files: {len(physical_files)}")
display(
    pd.DataFrame(
        {
            "audit": list(state.audits),
            "rows": [len(value) for value in state.audits.values()],
        }
    )
)

## Release gate

Seed v05 must not be generated until:

1. Required audit files are completed.
2. Register, template-family, and linguistic validation subsets have independent second annotations.
3. Agreement and detector metrics meet accepted decision-record thresholds.
4. Every sampling and exclusion decision is marked accepted.
5. The gated sampler validates the accepted decision record and persistent v05.1 manifest.